# Task 4 - Regional Node Candidate Screening

This notebook profiles the CoStar candidate facility exports, tags direct availability from `Building Status`, writes the reduced capacity/location dataset, and produces first-pass statistics for regional hub screening.

Task 4 distinguishes three site roles:

- Direct existing facilities: `Building Status == Existing`
- Pipeline/proxy facilities: `Under Construction` or `Final Planning`
- Primary regional hub candidates: direct existing logistics facilities with at least 20,000 sq ft of listed usable available space and valid coordinates


## Availability Rule

The status-based availability tag is:

$$
a_i = \mathbb{1}[\text{BuildingStatus}_i = \text{Existing}]
$$

The first-pass regional hub screen adds the Task 4 capacity floor:

$$
p_i = a_i \cdot \mathbb{1}[\text{UsableAvailableSF}_i \ge 20{,}000] \cdot \mathbb{1}[\text{Type}_i \in \text{Logistics}] \cdot \mathbb{1}[\text{valid coordinates}_i]
$$

`Existing` means directly usable by status. `Under Construction` and `Final Planning` remain in the data as pipeline/proxy candidates, but they are not primary regional hub candidates in the first group.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / "Data").exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Task4.task4_preprocess import run_pipeline, print_run_summary

pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", "{:,.2f}".format)


## Run Preprocessing

The raw CoStar exports remain unchanged in `Data/Task4/ALL`. Derived files are written to `Data/Task4/processed`, direct existing per-state copies are written to `Data/Task4/Available`, and the HTML map is written to `Data/Task4/figures`.


In [ ]:
candidates, outputs = run_pipeline()
print_run_summary(candidates, outputs)


## Overall Candidate Statistics


In [ ]:
summary = {
    "all_candidates": len(candidates),
    "direct_existing_by_status": int(candidates["is_directly_usable_by_status"].sum()),
    "pipeline_or_proxy_by_status": int(candidates["is_pipeline_or_proxy_by_status"].sum()),
    "listed_available_space": int(candidates["has_listed_available_space"].sum()),
    "primary_regional_hub_candidates": int(candidates["is_primary_regional_hub_candidate"].sum()),
    "total_usable_available_sf": candidates["usable_available_space_sf"].sum(),
    "median_rba_sf": candidates["rba_sf"].median(),
    "median_loading_docks": candidates["number_loading_docks"].median(),
}

print("Overall candidate statistics")
print("-" * 80)
for key, value in summary.items():
    label = key.replace("_", " ").title()
    if isinstance(value, float):
        print(f"{label:40s}: {value:,.2f}")
    else:
        print(f"{label:40s}: {value:,}")


In [ ]:
state_stats = pd.read_csv(PROJECT_ROOT / "Data/Task4/processed/state_candidate_stats.csv")
print("State candidate summary")
print("-" * 80)
print(state_stats.to_string(index=False))


In [ ]:
status_summary = pd.read_csv(PROJECT_ROOT / "Data/Task4/processed/status_summary.csv")
secondary_summary = pd.read_csv(PROJECT_ROOT / "Data/Task4/processed/secondary_type_summary.csv")

print("Availability status summary")
print("-" * 80)
print(status_summary.to_string(index=False))
print()
print("Secondary type summary")
print("-" * 80)
print(secondary_summary.to_string(index=False))


## Region Coverage Check

Candidates are spatially joined to the Task 3 county layer, then merged to Task 3 region ids. Regions with zero primary candidates should be handled in the next Task 4 pass using relaxed filters or proxy sites.


In [ ]:
region_stats_path = PROJECT_ROOT / "Data/Task4/processed/region_candidate_stats.csv"
region_stats = pd.read_csv(region_stats_path)
coverage = {
    "total_task3_regions": int(region_stats["region_id"].nunique()),
    "regions_with_any_candidate": int((region_stats["all_candidates"] > 0).sum()),
    "regions_without_any_candidate": int((region_stats["all_candidates"] == 0).sum()),
    "regions_with_primary_candidate": int((region_stats["primary_regional_hub_candidates"] > 0).sum()),
    "regions_without_primary_candidate": int((region_stats["primary_regional_hub_candidates"] == 0).sum()),
}
print("Region coverage summary")
print("-" * 80)
for key, value in coverage.items():
    print(f"{key.replace('_', ' ').title():40s}: {value:,}")

print()
print("Regions needing fallback/proxy treatment")
print("-" * 80)
gap_regions = region_stats[region_stats["primary_regional_hub_candidates"].eq(0)]
print(gap_regions.to_string(index=False))


## Candidate Map

The static PNG map colors candidates by availability class and scales points by listed usable available square footage.


In [ ]:
from IPython.display import Image

png_map_path = PROJECT_ROOT / "Data/Task4/figures/all_candidate_facilities_map.png"
print(f"PNG candidate map: {png_map_path.relative_to(PROJECT_ROOT)}")
Image(filename=str(png_map_path))


## Preprocessed Capacity/Location Dataset

This is the reduced dataset requested for downstream regional hub screening. It keeps candidate id/name, location fields, capacity fields, and Task 4 tags only.


In [ ]:
preprocessed = pd.read_csv(PROJECT_ROOT / "Data/Task4/processed/preprocessed_capacity_location.csv")
print("Preprocessed capacity/location dataset")
print("-" * 80)
print(f"Rows: {len(preprocessed):,}")
print(f"Columns: {preprocessed.shape[1]:,}")
print()
print("First 10 rows")
print("-" * 80)
print(preprocessed.head(10).to_string(index=False))
